In [ ]:
import pandas as pd
import torch 
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision 
import torchvision.transforms as transforms
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import requests
import json
import ast
import os

import difflib
import random
from tqdm import tqdm

https://www.kaggle.com/competitions/understanding_cloud_organization/

Notebooks:
- https://www.kaggle.com/code/artgor/segmentation-in-pytorch-using-convenient-tools
- https://www.kaggle.com/code/dhananjay3/image-segmentation-from-scratch-in-pytorch

In [ ]:
device = 'mps'

### Load Clouds Dataframe

In [ ]:
DATASET_PATH = "/Users/emulie/Downloads/understanding_cloud_organization/"

In [ ]:
df_train = pd.read_csv(DATASET_PATH + 'train.csv')

In [ ]:
df_train.head()

In [ ]:
labels = [label for img, label in df_train['Image_Label'].str.split('_')]

In [ ]:
df_train.shape

In [ ]:
torch.manual_seed(42)

### EDA

- How many empty masks?
- Class Count?

https://www.kaggle.com/code/aleksandradeis/understanding-clouds-eda

In [ ]:
df_train[['Image', 'Label']] = df_train['Image_Label'].str.split('_', expand=True)

In [ ]:
print(f"How many training samples: {len(df_train)}")
print(f"How many different images: {len(df_train['Image'].unique())}")
print(f"How many rows with empty mask: {df_train['EncodedPixels'].isna().sum()} ({round(100*df_train['EncodedPixels'].isna().sum() / len(df_train), 2)}%)")

In [ ]:
df_train['Label'].value_counts()

In [ ]:
fish = df_train[df_train['Label'] == 'Fish'].EncodedPixels.count()
flower = df_train[df_train['Label'] == 'Flower'].EncodedPixels.count()
gravel = df_train[df_train['Label'] == 'Gravel'].EncodedPixels.count()
sugar = df_train[df_train['Label'] == 'Sugar'].EncodedPixels.count()

print('There are {} fish clouds'.format(fish))
print('There are {} flower clouds'.format(flower))
print('There are {} gravel clouds'.format(gravel))
print('There are {} sugar clouds'.format(sugar))

labels = ['Fish', 'Flower', 'Gravel', 'Sugar']
sizes = [fish, flower, gravel, sugar]
fig, ax = plt.subplots(figsize=(2, 2))
ax.pie(sizes, labels=labels, autopct='%1.1f%%', shadow=True, startangle=90)
ax.axis('equal')
ax.set_title('Cloud Types')
plt.show()

In [ ]:
## correlation between cloud type
corr_df = pd.get_dummies(df_train, columns = ['Label'], dtype=int)
corr_df = corr_df[~pd.isna(corr_df['EncodedPixels'])].groupby('Image').sum()[['Label_Fish', 'Label_Flower', 'Label_Gravel', 'Label_Sugar']]
corrs = np.corrcoef(corr_df.values.T)

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(4,4)})
hm=sns.heatmap(corrs, cbar = True, annot=True, square = True, fmt = '.2f',
              yticklabels = ['Fish', 'Flower', 'Gravel', 'Sugar'], 
               xticklabels = ['Fish', 'Flower', 'Gravel', 'Sugar']).set_title('Cloud type correlation heatmap')

fig = hm.get_figure()

In [ ]:
## making the mask

df_train

In [ ]:
t = df_train.loc[(df_train['Image'] == 'ffea4f4.jpg') & (df_train['Label'] == 'Fish'), 'EncodedPixels'].values[0]

In [ ]:
pd.isna(t)

### Making the Cloud Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np


# Only for Train and Validation
class CloudDataset(Dataset):
    def __init__(self, dataset_path: str, split: str, train_split: float = 0.8) -> None:
        self.dataset_path = dataset_path
        self.split = split
        self.train_split = train_split

        self.df_train, self.image_ids = self._load_train_csv()

    def _load_train_csv(self):
        csv_path = self.dataset_path + "train.csv"
        try:
            df_train = pd.read_csv(csv_path)
        except Exception as e:
            raise FileNotFoundError(f"Couldn't find {csv_path}. Please check {e}")

        df_train[["Image", "Label"]] = df_train["Image_Label"].str.split(
            "_", expand=True
        )
        image_ids = df_train["Image"].unique()
        n = len(image_ids)
        e_train = int(n * self.train_split)
        if self.split == 'train':
            return df_train, image_ids[:e_train]
        elif self.split == 'val':
            return df_train, image_ids[e_train:]
        else:
            raise ValueError(f"Split not implemented. Please select 'train' or 'test'")
            

    def __len__(self):
        return len(self.image_ids)

    def _get_mask(self, idx, shape=(1400, 2100)):
        """
        Masks: 'Fish', 'Flower', 'Gravel', 'Sugar',
        """
        height, width = shape
        file_name = self.image_ids[idx]
        masks = []
        for label in ["Fish", "Flower", "Gravel", "Sugar"]:
            image_mask = self.df_train["Image"] == file_name
            label_mask = self.df_train["Label"] == label
            encoded_pixels = self.df_train.loc[
                image_mask & label_mask, "EncodedPixels"
            ].values[0]
            
            if pd.isna(encoded_pixels):
                masks.append(np.zeros((width, height), dtype=np.uint8).T)
                # masks.append(np.zeros((width, height), dtype=np.uint8))
                continue

            encoded_pixels = list(map(int, encoded_pixels.split()))
            mask = np.zeros(width * height, dtype=np.uint8)
            for i in range(0, len(encoded_pixels), 2):
                start, length = encoded_pixels[i] - 1, encoded_pixels[i+1]
                mask[start:start+length] = 1
            mask = mask.reshape((width, height)).T
            # mask = mask.reshape((width, height))
            masks.append(mask)
        return np.stack(masks, axis=0)

    def __getitem__(self, idx):
        sub_directory = "train_images/"
        image = Image.open(self.dataset_path + sub_directory + self.image_ids[idx])
        img_array = np.array(image)
        mask = self._get_mask(idx)

        if img_array.ndim == 4:
            # batch
            img_array = np.transpose(img_array, (0, 3, 1, 2))
        elif img_array.ndim == 3:
            # single image
            img_array = np.transpose(img_array, (2, 0, 1))
        else:
            raise ValueError(f"Unexpected shape: {img_array.shape}")

        # return {"image": img_array, "mask": mask} # np.transpose(img_array, (0, 3, 1, 2))
        return {"image": img_array, "mask": mask} # 


In [ ]:
train_dataset = CloudDataset(DATASET_PATH, 'train', train_split=0.8)
test_dataset = CloudDataset(DATASET_PATH, 'val', train_split=0.8)

TRAIN_BATCH_SIZE, TEST_BATCH_SIZE = 8, 8
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=True)

In [ ]:
# # Note: mask should be shape: (width, height, 4) 
# # DEPRECATED --
# class CloudDataset(Dataset):
    
#     def __init__(self, dataset_path: str, split: str, train_split: float = 0.8):
#         self.dataset_path = dataset_path
#         self.split = split
#         self.train_split = train_split
        
#         self.image_files, self.labels, self.encoded_pixels = self._load_image_files()
        
#     def _load_image_files(self):
#         # read csv file
#         df_train = pd.read_csv(self.dataset_path + 'train.csv')
        
#         # read image metadata
#         n = df_train.shape[0]
#         e_train = int(n * self.train_split)
#         df_train['label'] = df_train['Image_Label'].apply(lambda x: x.split('_')[1])
#         df_train['img_file'] = df_train['Image_Label'].apply(lambda x: x.split('_')[0])
        
#         image_files = df_train['img_file'].tolist()
#         labels = df_train['label'].tolist()
#         encoded_pixels = [pixels if not pd.isna(pixels) else [] for pixels in df_train['EncodedPixels'].tolist()]
#         # masks = [self._get_mask(pixels) for pixels in df_train['EncodedPixels'].tolist()]
        
#         if self.split == 'train':
#             return image_files[:e_train], labels[:e_train], encoded_pixels[:e_train]
#         elif self.split == 'test':
#             return image_files[e_train:], labels[e_train:], encoded_pixels[e_train:]
#         else:
#             raise ValueError(f"Split not implemented. Please select 'train' or 'test'")
    
#     def _get_mask(self, pixels, shape = (1400, 2100)):  #FIXME?
#         height, width = shape[0], shape[1]
#         if pixels == []:
#             return np.zeros((width, height)).T

#         pixels = list(map(int, pixels.split()))
    
#         mask =  np.zeros(height * width, dtype=np.uint8)
#         for i in range(0, len(pixels) - 1, 2):
#             start, length = pixels[i] - 1, pixels[i+1]
#             mask[start:start+length] = 1
#         mask = mask.reshape((width, height)).T
#         return mask
        
            
#     def __len__(self):
#         return len(self.image_files)
    
#     def __getitem__(self, idx):
#         # sub_directory = 'train_images/' if self.split == 'train' else 'test_images/'
#         sub_directory = 'train_images/'
#         image = Image.open(self.dataset_path + sub_directory + self.image_files[idx])
#         img_array = np.asarray(image)
#         mask = self._get_mask(self.encoded_pixels[idx])
        
#         return {'image': img_array, 'label': self.labels[idx], 
#                 'mask': mask}


In [ ]:
# train_dataset = CloudDataset(DATASET_PATH, 'train', train_split=0.8)
# test_dataset = CloudDataset(DATASET_PATH, 'val', train_split=0.8)

# TRAIN_BATCH_SIZE, TEST_BATCH_SIZE = 8, 8
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True)

In [ ]:
# len(train_dataset)

In [ ]:
batch = next(iter(train_loader))

In [ ]:
batch['image'].size() # # torch.Size([1, 1400, 2100, 3])

In [ ]:
batch['mask'].size()

In [ ]:
# batch['mask'][0, 0, :, :].shape

In [ ]:
# batch['image'][0].shape

### Visualize Cloud

In [ ]:
batch = next(iter(train_loader))
# batch

In [ ]:
def show_batch(batch, batch_size: int):
    labels = ['Fish', 'Flower', 'Gravel', 'Sugar']
    for i in range(batch_size):
        fig, axes = plt.subplots(1, len(labels)+1, figsize=(10,8))
        axes[0].imshow(np.transpose(batch['image'][i], (1, 2, 0)))
        for k, label in enumerate(labels):
            axes[k+1].imshow(batch['mask'][i, k, :, :], alpha=0.75)
            axes[k+1].set_title(f"{label} - {batch['mask'][i, k, :, :].any()}")
        plt.tight_layout()
        plt.show()
        
show_batch(batch, batch_size=TRAIN_BATCH_SIZE)

In [ ]:
# def show_batch(batch, batch_size: int):
#     for i in range(batch_size):
#         plt.imshow(batch['image'][i])
#         plt.imshow(batch['mask'][i], alpha=0.5)
#         plt.title(batch['label'][i ])
#         plt.show()
# show_batch(batch, batch_size=TRAIN_BATCH_SIZE)